# Step 1: Feature Extraction & Vectorization




Importing and loading



In [12]:
import pandas as pd

In [32]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer #  Vectorization using scikit-learn

from sklearn.metrics.pairwise import cosine_similarity
# Cosine Similarity Matching

import numpy as np


In [19]:

resume_df = pd.read_csv('sample_data/resume_clean.csv').dropna(subset=['clean_description'])
job_df = pd.read_csv('sample_data/job_clean.csv').dropna(subset=['clean_description'])

resumes = resume_df['clean_description'].values
jobs = job_df['clean_description'].values

feature extraction

In [21]:
# Initialize vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

# Fit on a combined corpus for a shared vocabulary space
combined_text = list(resumes) + list(jobs)
vectorizer.fit(combined_text)

# Transform the individual corpora
resume_tfidf = vectorizer.transform(resumes)
job_tfidf = vectorizer.transform(jobs)

print(f"Resume matrix shape {resume_tfidf.shape}")
print(f"job matrix shape {job_tfidf.shape}")



Resume matrix shape (2483, 5000)
job matrix shape (853, 5000)


## Step 2: Implement Similarity Matching



this function gets cosine similarity between a specific resume and all job descriptions, and return the top matching jobs


In [35]:

def get_top_matches(resume_idx, resume_tfidf, job_tfidf, jobs_df, top_n=5):
    # calculate cosine similarity for one resume against the jobs
    similarities = cosine_similarity(resume_tfidf[resume_idx], job_tfidf).flatten()

    # indices of top N highest similarity scores
    top_indices = similarities.argsort()[::-1][:top_n]

    results = []
    for idx in top_indices:
        #  extract and slice the job description snippet
        job_desc = str(jobs_df.iloc[idx].get('clean_description', ''))
        snippet = job_desc[:300] + '...' if len(job_desc) > 300 else job_desc

        results.append({
            'job index': idx,
            'similarity score': similarities[idx],
            'job description': snippet
        })

    return pd.DataFrame(results)



# Step 3: Evaluation to validate the performance of the matching pipeline.


this function gets a baseline Precision@K metric evaluating whether recommended jobs share the same category or required alignment as the candidate resume

In [38]:

def evaluate_precision_at_k(resume_tfidf, job_tfidf, resume_df, job_df, k=5):

    precisions = []

    for i in range(min(len(resume_df), 50)): # evaluate on a sample subset
        # gets top K job indices
        sims = cosine_similarity(resume_tfidf[i], job_tfidf).flatten()
        top_indices = sims.argsort()[::-1][:k]

        # check if categories or degrees align between resume and job
        resume_category = resume_df.iloc[i].get('Category', None)

        if resume_category and 'position_title' in job_df.columns:
            # counts how many of the top K jobs match the resume category context
            relevant_count = sum(1 for idx in top_indices if resume_category.lower() in str(job_df.iloc[idx]['position_title']).lower())
            precisions.append(relevant_count / k)

    avg_precision = np.mean(precisions) if precisions else 0.0
    print(f"the mean precision at top {k} recommended jobs is {avg_precision:.4f}")
    return avg_precision


run the evaluation

In [39]:

evaluate_precision_at_k(resume_tfidf, job_tfidf, resume_df, job_df, k=5)

the mean precision at top 5 recommended jobs is 0.1880


np.float64(0.188)

# Step 4: Finalize Deliverables


we built a web app to compute the similarities and display the top 5 matching job roles along with their match scores and text snippets

In [41]:
# this saves an app.py template that you can interact with on the web
streamlit_app_code = """
import streamlit as st
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

st.title("Job Market Intelligence: Resume Matcher")
st.write("Upload or paste a candidate resume to find top-matching job listings.")

user_resume = st.text_area("Paste Resume Text Here:")

if st.button("Find Matching Jobs"):
    if user_resume.strip():
        job_df = pd.read_csv('job_clean.csv').dropna(subset=['clean_description'])

        # Quick inference vectorization
        vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
        job_vectors = vectorizer.fit_transform(job_df['clean_description'])
        user_vec = vectorizer.transform([user_resume])

        similarities = cosine_similarity(user_vec, job_vectors).flatten()
        top_indices = similarities.argsort()[::-1][:5]

        st.subheader("Top Matching Roles:")
        for idx in top_indices:
            st.markdown(f"**Score:** {similarities[idx]:.4f}")
            st.write(job_df.iloc[idx]['clean_description'][:400] + "...")
            st.markdown("---")
    else:
        st.warning("Please enter some resume text first.")
"""

with open("app.py", "w") as f:
    f.write(streamlit_app_code.strip())

